# 🎓 GPA Assistant — RAG + LLM Integrated Backend

**Government Polytechnic Adityapur | Jamshedpur, Jharkhand**

---

This notebook is the **Python backend** for the GPA chatbot system. It:

1. **Ingests** scanned notice images from Google Drive using Gemini Vision (VLM)
2. **Chunks** and **embeds** the extracted text using BAAI/bge-m3 multilingual embeddings
3. **Stores** vectors in ChromaDB + BM25 index for hybrid retrieval
4. **Serves** a Flask API that the frontend chatbot.js can call for context-aware answers
5. **Exposes** a public ngrok URL so the browser-based chatbot can reach this Colab backend

### Architecture
```
Browser (chatbot.js)  ──POST /api/rag/query──►  Flask Server (this notebook)
                                                        │
                                              HybridRetriever (ChromaDB + BM25)
                                                        │
                                              Gemini 2.5 Flash (answer generation)
                                                        │
                      ◄──── { answer, sources } ────────┘
```

### How to use
1. Run **Section 1** (setup) once per session
2. Run **Section 2** (ingest) whenever you add new notice images to Drive
3. Run **Section 3** (server) to start the API — copy the ngrok URL into `chatbot.js`

---
## ⚙️ SECTION 1 — Setup & Configuration

In [ ]:
# 1.1  Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted.')

In [ ]:
# 1.2  Install all dependencies
!pip install -q -U \
    google-genai \
    langchain==0.2.0 \
    langchain-community==0.2.0 \
    langchain-core==0.2.0 \
    langchain-text-splitters \
    langchain-huggingface \
    chromadb \
    rank_bm25 \
    flask \
    flask-cors \
    pyngrok \
    pydantic

print('✅ All dependencies installed.')

In [ ]:
# 1.3  Configuration — EDIT THESE BEFORE RUNNING
import os, json

GOOGLE_API_KEY   = 'YOUR_GOOGLE_API_KEY_HERE'   # Gemini API key
NGROK_AUTH_TOKEN = 'YOUR_NGROK_AUTH_TOKEN'       # Free at https://dashboard.ngrok.com

# Paths (all on Google Drive)
BASE_PATH      = '/content/drive/MyDrive/RAG'
INPUT_PATH     = f'{BASE_PATH}/input_docs'
PROCESSED_PATH = f'{BASE_PATH}/processed_docs'
CHROMA_PATH    = f'{BASE_PATH}/chroma_db'
LOG_PATH       = f'{BASE_PATH}/processed_log.json'
BM25_DOCS_PATH = f'{BASE_PATH}/bm25_docs.json'

for path in [INPUT_PATH, PROCESSED_PATH, CHROMA_PATH]:
    os.makedirs(path, exist_ok=True)

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
print('✅ Configuration complete.')
print(f'   Input docs  → {INPUT_PATH}')
print(f'   ChromaDB    → {CHROMA_PATH}')

---
## 📄 SECTION 2 — Document Ingestion Pipeline

Place your scanned notice images (`.jpg`, `.jpeg`, `.png`) inside:
```
Google Drive → MyDrive → RAG → input_docs/
```
Then run all cells in this section.

In [ ]:
# 2.1  Initialise Gemini Client
from PIL import Image
from google import genai
from google.genai import types

client = genai.Client()
print('✅ Gemini client ready.')

In [ ]:
# 2.2  Pydantic schema for structured VLM extraction
from pydantic import BaseModel, Field
from typing import List, Optional

class SignatoryEntity(BaseModel):
    name_or_designation: str = Field(description="Official designation like 'प्राचार्य'")
    organization: Optional[str] = Field(description='Institution name if mentioned')
    date_signed:  Optional[str] = Field(description='Date near signature if visible')

class AcademicNoticeSchema(BaseModel):
    issuing_authority:  Optional[str]           = None
    reference_number:   Optional[str]           = None
    date_issued:        Optional[str]           = None
    subject_line:       Optional[str]           = None
    target_audience:    Optional[List[str]]     = None
    main_body_content:  Optional[str]           = None
    signatories:        Optional[List[SignatoryEntity]] = None
    distribution_list:  Optional[List[str]]     = None
    document_type:      Optional[str]           = None
    extra_fields:       Optional[str]           = None

print('✅ Pydantic schema defined.')

In [ ]:
# 2.3  VLM Extraction Module
EXTRACTION_PROMPT = '''
You are an intelligent document understanding system for academic/administrative documents.

STEP 1 — Classify document: notice | office_order | exam_schedule | email | tabular_data | unknown
STEP 2 — HIGH PRIORITY: extract reference_number, date_issued, issuing_authority accurately.
STEP 3 — main_body_content MUST contain ALL readable text. Never return null here.
STEP 4 — Preserve Hindi and English exactly. Tables in Markdown. No hallucination.
STEP 5 — extra_fields: JSON string for tables or unknown structures.
Return valid JSON only.
'''

def extract_structured_metadata(image_path: str) -> AcademicNoticeSchema:
    try:
        document_image = Image.open(image_path)
    except FileNotFoundError:
        raise ValueError(f'Image not found: {image_path}')

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[document_image, EXTRACTION_PROMPT],
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=AcademicNoticeSchema,
            temperature=0.0
        )
    )
    parsed = response.parsed
    if not parsed.main_body_content:
        parsed.main_body_content = ''
    return parsed

print('✅ VLM extraction module ready.')

In [ ]:
# 2.4  Chunking Module
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def create_intelligent_context_chunks(structured_notice: AcademicNoticeSchema) -> List[Document]:
    global_metadata = {
        'issuing_authority': structured_notice.issuing_authority,
        'reference_number':  structured_notice.reference_number or 'UNKNOWN_REF',
        'date_issued':       structured_notice.date_issued,
        'subject':           structured_notice.subject_line or 'General Administrative Notice'
    }

    context_header = (
        f"Notice Reference: {global_metadata['reference_number']}, "
        f"Date: {global_metadata['date_issued']}, "
        f"Subject: {global_metadata['subject']}.\nContent: "
    )

    body_text = (structured_notice.main_body_content or '').strip()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=250,
        separators=['\n\n', '\n', '।', '.', ' ', '']
    )

    return [
        Document(page_content=context_header + chunk, metadata=global_metadata)
        for chunk in text_splitter.split_text(body_text)
    ]

print('✅ Chunking module ready.')

In [ ]:
# 2.5  Multilingual Embedding Model (BAAI/bge-m3)
# NOTE: First load takes ~2-3 minutes to download the model weights.
from langchain_huggingface import HuggingFaceEmbeddings

def initialize_embedding_model() -> HuggingFaceEmbeddings:
    return HuggingFaceEmbeddings(
        model_name='BAAI/bge-m3',
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )

print('⏳ Loading BAAI/bge-m3 embedding model (first run ~2-3 min)...')
embedding_model = initialize_embedding_model()
print('✅ Embedding model loaded.')

In [ ]:
# 2.6  Hybrid Retrieval System (ChromaDB + BM25)
from langchain.vectorstores import Chroma
from langchain.retrievers import BM25Retriever, EnsembleRetriever

def load_processed_log():
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, 'r') as f:
            return json.load(f)
    return []

def save_processed_log(log):
    with open(LOG_PATH, 'w') as f:
        json.dump(log, f)

def load_bm25_docs():
    if os.path.exists(BM25_DOCS_PATH):
        with open(BM25_DOCS_PATH, 'r') as f:
            return json.load(f)
    return []

def save_bm25_docs(docs):
    with open(BM25_DOCS_PATH, 'w') as f:
        json.dump(docs, f, indent=2)

def build_hybrid_retriever(new_documents, embeddings):
    vector_store = Chroma(
        persist_directory=CHROMA_PATH,
        embedding_function=embeddings
    )
    if new_documents:
        vector_store.add_documents(new_documents)
        vector_store.persist()

    semantic_retriever = vector_store.as_retriever(search_kwargs={'k': 4})

    existing_docs = load_bm25_docs()
    new_doc_dicts = [{'content': d.page_content, 'metadata': d.metadata} for d in new_documents]
    all_docs = existing_docs + new_doc_dicts
    save_bm25_docs(all_docs)

    retrievers = [semantic_retriever]
    weights    = [1.0]

    if all_docs:
        bm25_retriever   = BM25Retriever.from_texts([d['content'] for d in all_docs])
        bm25_retriever.k = 4
        retrievers.append(bm25_retriever)
        weights = [0.6, 0.4]

    return EnsembleRetriever(retrievers=retrievers, weights=weights)

print('✅ Hybrid retrieval module ready.')

In [ ]:
# 2.7  Run Document Ingestion
import shutil

processed_files = load_processed_log()
all_files       = os.listdir(INPUT_PATH)
new_files       = [
    f for f in all_files
    if f not in processed_files and f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

print(f'🔍 Found {len(new_files)} new file(s): {new_files}')

documents_to_add = []

for i, filename in enumerate(new_files, 1):
    image_path = os.path.join(INPUT_PATH, filename)
    print(f'  [{i}/{len(new_files)}] Processing: {filename}')
    try:
        structured_notice = extract_structured_metadata(image_path)
        chunks = create_intelligent_context_chunks(structured_notice)
        documents_to_add.extend(chunks)
        shutil.move(image_path, os.path.join(PROCESSED_PATH, filename))
        processed_files.append(filename)
        print(f'     ✅ {len(chunks)} chunk(s) extracted')
    except Exception as e:
        print(f'     ❌ Failed: {e}')

save_processed_log(processed_files)

print('\n⏳ Building hybrid retriever...')
hybrid_retriever = build_hybrid_retriever(documents_to_add, embedding_model)
print(f'\n✅ Done. New chunks: {len(documents_to_add)}, Total BM25: {len(load_bm25_docs())}')

---
## 🚀 SECTION 3 — Flask API Server

**Endpoints:**

| Method | Path | Description |
|--------|------|-------------|
| `POST` | `/api/rag/query` | RAG query → AI answer + sources |
| `GET`  | `/api/rag/stats` | KB statistics |
| `GET`  | `/api/health`    | Health check |

After running the cell below, **copy the ngrok URL** and paste it into `chatbot.js`:
```javascript
RAG_BACKEND_URL: 'https://xxxx-xx-xx.ngrok-free.app',
```

In [ ]:
# 3.1  RAG Query Function
GPA_SYSTEM_PROMPT = '''
You are "GPA Assistant" — an intelligent chatbot for
Government Polytechnic Adityapur (GPA), Jamshedpur, Jharkhand.

RULES:
- Answer ONLY using the provided CONTEXT when relevant.
- If context lacks the answer, say: "Information not available in knowledge base."
- Keep answers concise and factual. Preserve Hindi text as-is.
- Do NOT hallucinate dates, numbers, or names.
'''

def generate_rag_answer(user_query: str, retriever) -> dict:
    retrieved_docs = retriever.invoke(user_query)

    if not retrieved_docs:
        return {
            'answer': 'Information regarding this query is not available in our knowledge base.',
            'sources': [],
            'chunk_count': 0
        }

    context_blocks = []
    sources = []
    for i, doc in enumerate(retrieved_docs, 1):
        context_blocks.append(f'[Source {i}]: {doc.page_content}')
        meta = doc.metadata
        sources.append({
            'index':     i,
            'reference': meta.get('reference_number', 'N/A'),
            'date':      meta.get('date_issued', 'N/A'),
            'subject':   meta.get('subject', 'N/A'),
            'authority': meta.get('issuing_authority', 'N/A')
        })

    context = '\n\n---\n\n'.join(context_blocks)
    prompt = f'''{GPA_SYSTEM_PROMPT}

CONTEXT:
{context}

USER QUERY: {user_query}

Answer:'''

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2)
    )

    return {'answer': response.text, 'sources': sources, 'chunk_count': len(retrieved_docs)}

print('✅ RAG query function ready.')

In [ ]:
# 3.2  Start Flask API Server with ngrok
import threading
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok, conf

app = Flask(__name__)
CORS(app, resources={r'/api/*': {'origins': '*'}})

_retriever = hybrid_retriever

@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'service': 'GPA RAG Backend', 'version': '2.0'})

@app.route('/api/rag/stats', methods=['GET'])
def stats():
    try:
        return jsonify({
            'total_chunks':    len(load_bm25_docs()),
            'total_documents': len(load_processed_log()),
            'status':          'ready'
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/rag/query', methods=['POST'])
def rag_query():
    if _retriever is None:
        return jsonify({'error': 'RAG not initialised. Run Section 2 first.'}), 503
    data = request.get_json(silent=True)
    if not data or 'query' not in data:
        return jsonify({'error': 'Missing "query" field.'}), 400
    user_query = str(data['query']).strip()
    if not user_query:
        return jsonify({'error': 'Query cannot be empty.'}), 400
    try:
        return jsonify(generate_rag_answer(user_query, _retriever))
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# Start ngrok tunnel
conf.get_default().auth_token = NGROK_AUTH_TOKEN
ngrok.kill()
public_url = ngrok.connect(5000).public_url

print('=' * 60)
print('🌐  GPA RAG Backend is LIVE!')
print('=' * 60)
print(f'   Public URL : {public_url}')
print()
print('📋  Paste into chatbot.js:')
print(f'    RAG_BACKEND_URL: \'{public_url}\',')
print()
print(f'   GET  {public_url}/api/health')
print(f'   GET  {public_url}/api/rag/stats')
print(f'   POST {public_url}/api/rag/query')
print('=' * 60)

threading.Thread(target=lambda: app.run(port=5000, debug=False, use_reloader=False), daemon=True).start()
print('\n⚡ Server running. Keep this notebook open.')

---
## 🔧 SECTION 4 — Utilities

In [ ]:
# 4.1  Test a query directly
test_query = 'What is the information about E-Kalyan scholarship?'
result = generate_rag_answer(test_query, hybrid_retriever)
print(f'Query   : {test_query}')
print(f'Chunks  : {result["chunk_count"]}')
print(f'Sources : {result["sources"]}')
print()
print('─── Answer ───')
print(result['answer'])

In [ ]:
# 4.2  Add a plain-text document manually (no image needed)
def add_text_document(title, content, date='N/A', ref='MANUAL'):
    global hybrid_retriever
    splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=250)
    chunks   = splitter.split_text(content)
    header   = f'Notice Reference: {ref}, Date: {date}, Subject: {title}.\nContent: '
    metadata = {'reference_number': ref, 'date_issued': date, 'subject': title, 'issuing_authority': 'Manual'}
    docs = [Document(page_content=header + c, metadata=metadata) for c in chunks]
    hybrid_retriever = build_hybrid_retriever(docs, embedding_model)
    print(f'✅ Added "{title}" with {len(docs)} chunk(s).')

# Example (uncomment to use):
# add_text_document(
#     title='E-Kalyan Scholarship 2026',
#     content='Students from SC/ST/OBC must apply at ekalyan.cgg.gov.in before March 31, 2026.',
#     date='2026-02-01',
#     ref='EKalyan-2026'
# )
print('✅ add_text_document() helper ready.')

In [ ]:
# 4.3  Export / Import knowledge base snapshot
EXPORT_PATH = f'{BASE_PATH}/kb_export.json'

def export_knowledge_base():
    snapshot = {'bm25_docs': load_bm25_docs(), 'processed_files': load_processed_log()}
    with open(EXPORT_PATH, 'w') as f:
        json.dump(snapshot, f, indent=2, ensure_ascii=False)
    size_kb = os.path.getsize(EXPORT_PATH) / 1024
    print(f'✅ Exported to {EXPORT_PATH}  ({size_kb:.1f} KB)')

def import_knowledge_base(path=EXPORT_PATH):
    global hybrid_retriever
    with open(path, 'r') as f:
        snapshot = json.load(f)
    save_bm25_docs(snapshot['bm25_docs'])
    save_processed_log(snapshot['processed_files'])
    hybrid_retriever = build_hybrid_retriever([], embedding_model)
    print(f'✅ Imported from {path}')
    print(f'   Chunks: {len(snapshot["bm25_docs"])}, Files: {len(snapshot["processed_files"])}')

# Uncomment to use:
# export_knowledge_base()
# import_knowledge_base()
print('✅ Export/Import helpers ready.')

In [ ]:
# 4.4  Stop the server (run when done)
# ngrok.kill()
# print('🔴 Server stopped.')